# Fine-Tuning de BERT — IMDB Sentiment

> Parte da série [ML Notebooks](../README.md) — por **Nandobez**.


## Intuição

BERT é uma pilha de blocos encoder pré-treinada com masked language modeling e next-sentence prediction. Para classificação, adicionamos uma cabeça pequena sobre a representação do `[CLS]` e fine-tune o modelo inteiro na tarefa alvo — aqui, sentimento binário no IMDB.


## Formulação Matemática

Para uma sequência de entrada usamos `[CLS] tokens [SEP]`. O último estado oculto na posição 0 é `h_{[CLS]}` e os logits são

$$\hat{y} = W\,h_{[CLS]} + b, \quad \mathcal{L} = \text{CrossEntropy}(\hat{y}, y).$$

Retropropagamos por todas as camadas; isso é *fine-tuning*, não extração de features.


## Implementação


In [ ]:
# Install once: pip install transformers datasets accelerate
import torch
from torch.utils.data import DataLoader
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, get_linear_schedule_with_warmup


In [ ]:
model_name = 'distilbert-base-uncased'
tok = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model.to(device)


In [ ]:
ds = load_dataset('imdb')

def encode(batch):
    return tok(batch['text'], truncation=True, padding='max_length', max_length=128)

train = ds['train'].shuffle(seed=0).select(range(2000)).map(encode, batched=True)
val   = ds['test'].shuffle(seed=0).select(range(500)).map(encode, batched=True)
train.set_format('torch', columns=['input_ids', 'attention_mask', 'label'])
val.set_format('torch', columns=['input_ids', 'attention_mask', 'label'])

train_dl = DataLoader(train, batch_size=16, shuffle=True)
val_dl   = DataLoader(val, batch_size=32)


In [ ]:
opt = torch.optim.AdamW(model.parameters(), lr=2e-5)
sched = get_linear_schedule_with_warmup(opt, num_warmup_steps=0, num_training_steps=len(train_dl) * 2)

for epoch in range(2):
    model.train()
    for batch in train_dl:
        batch = {k: v.to(device) for k, v in batch.items()}
        out = model(input_ids=batch['input_ids'], attention_mask=batch['attention_mask'], labels=batch['label'])
        out.loss.backward()
        opt.step(); sched.step(); opt.zero_grad()
    print(f'epoch {epoch} train loss {out.loss.item():.3f}')


## Experimento


In [ ]:
model.eval()
correct = total = 0
with torch.no_grad():
    for batch in val_dl:
        batch = {k: v.to(device) for k, v in batch.items()}
        logits = model(input_ids=batch['input_ids'], attention_mask=batch['attention_mask']).logits
        correct += (logits.argmax(-1) == batch['label']).sum().item()
        total += batch['label'].size(0)
print(f'val accuracy: {correct/total:.3f}')


## Discussão

- Use um learning rate bem baixo (2e-5) — BERT já está bem treinado e você não quer sobrescrever as features.
- Warmup + decaimento linear é o schedule padrão para fine-tuning.
- Para um treino real, use o IMDB completo e um max_length maior (256–512).


## Referências

- Repositório da série: [github.com/Nandobez/ml-notebooks](https://github.com/Nandobez/ml-notebooks)
- Autor: [Nandobez](https://github.com/Nandobez)
